# Experiment grid run (M5)

Thin driver, like the baseline notebook: all real code is in the repo (`src/longtail_lightgcn`). One session of this notebook does **one run** of the grid (`MAX_RUNS = 1`); it picks the next unfinished run automatically from the results committed to the repo.

**Before running:** turn on **Internet** and set the accelerator to a **GPU**. Then use **Save Version -> Save & Run All (Commit)** so it runs in the background and keeps the output.

**Every session:** set `STAGE` in step 4 (1 until stage 1 is finished and `results/dials.json` is committed, then 2).

**After the session:** download `results/` from the version's Output tab, copy it into your local repo, then `git add results/ && git commit && git push` (see the last cell).

## 1. Get the code and the results so far
Clones the repo (including the `results/` committed after earlier sessions). Don't change the training code in the middle of the grid; every row logs the commit it ran on.

In [ ]:
import os

REPO_URL = "https://github.com/rithikkulkarni/longtail-lightgcn.git"
COMMIT = ""  # optional: paste a commit hash to pin the exact code; empty = latest main

%cd /kaggle/working
if not os.path.exists("longtail-lightgcn"):
    !git clone $REPO_URL
%cd /kaggle/working/longtail-lightgcn
!git pull -q
if COMMIT:
    !git checkout -q $COMMIT
!pip install -e . --no-deps -q
!git rev-parse HEAD

## 2. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. A 200-epoch run will take many hours on CPU. Stop and fix the notebook settings.")

## 3. Build the data
Downloads ML-1M and writes `data/processed/` (splits 0-4 and `items.csv`).

In [ ]:
!python scripts/prepare_data.py

## 4. Run the next unfinished run
Copies the committed `results/` to `/kaggle/working/results` (the folder you will download), shows what is done, then runs `MAX_RUNS` unfinished runs of the stage. A run takes about 1.2 hours at 200 epochs.

In [ ]:
STAGE = 1     # 1 = baseline, C, and A / B at each dial value; 2 = the 8 on/off combinations (needs results/dials.json)
MAX_RUNS = 1  # unfinished runs to do this session; 1 = one run per Kaggle session (D-29)

import os, shutil
OUT = "/kaggle/working/results"
if os.path.isdir("results"):  # start from the results committed to the repo so finished runs are skipped
    shutil.copytree("results", OUT, dirs_exist_ok=True)
os.makedirs(OUT, exist_ok=True)

!python -m longtail_lightgcn.run_grid --stage $STAGE --list --results_dir $OUT
!python -m longtail_lightgcn.run_grid --stage $STAGE --max_runs $MAX_RUNS --results_dir $OUT

## 5. Look at what was written

In [ ]:
import json
import pandas as pd

rows = [json.loads(l) for l in open("/kaggle/working/results/results.jsonl") if l.strip()]
print(len(rows), "runs in results.jsonl")
pd.DataFrame([
    {"key": r["key"], "label": r["label"], "best_epoch": r["best_epoch"], "hours": round(r["train_seconds"] / 3600, 2),
     **{f"val_{k}": round(v, 4) for k, v in r["val"].items()}, **{f"test_{k}": round(v, 4) for k, v in r["test"].items()}}
    for r in rows
])

## 6. After the session: save the result to your repo
1. When the version finishes, open it and go to the **Output** tab.
2. Download **`results/`** (it holds `results.jsonl` and `recs/`). You do not need the rest of the output.
3. In your local repo, replace `results/results.jsonl` with the downloaded one (it contains all earlier rows plus the new one) and copy the new files from `recs/` into `results/recs/`.
4. `git add results/ && git commit -m "grid run: <key>" && git push`.
5. Start the next session. It will clone the repo, see the finished run, and do the next one.

When stage 1 is finished (8 runs): run `python -m longtail_lightgcn.select_dials --results_dir results` locally, review the choice, commit `results/dials.json`, then set `STAGE = 2`.